In [ ]:
pip install beautifulsoup4==4.12.3

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline


pipe = pipeline("text2text-generation", model="google/flan-t5-base")
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
## create a prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful school assistant"),
    ("human", "{user_input}")
])

def messages(user_input):
    return prompt.format_messages(user_input=user_input)



In [ ]:
## time to prepare my file.
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="Final_files/school_file.json",
    jq_schema=".faqs[] | .answer"
)
documents = loader.load()

## now we chunk
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,add_start_index=True)
allsplit=text_splitter.split_documents(documents)


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings


embed = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


vectorstore = Chroma.from_documents(
    documents=allsplit,
    embedding=embed,
    persist_directory="new_db"
)

vectorstore.persist()



In [ ]:
from langchain.tools import tool
import requests
from bs4 import BeautifulSoup
        
@tool
def search_school_website(url: str) :
    """Fetch and extract text from a school website."""
    
    response = requests.get("https://unilag.edu.ng/")
    soup = BeautifulSoup(response.text, "html.parser")
    
    text = soup.get_text()

def choose_which(query):
    result=vectorstore.similarity_search(query, k=1)
    if result:
        return result[0].page_content

In [ ]:
from langchain.agents import initialize_agent, AgentType

tools=[search_school_website]
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
    max_iteration=3
)

agent.run("where is unilag located")